# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Operating decision:** Which pages should an editor review first, and what should they check before acting?

Week 6 found that the shallow model beats the hand rule but does not beat the test base rate and is unstable across client groups. This playbook therefore uses a transparent, March-only rule to create a review queue. It is a triage system for human review, not an automated refresh decision.

## 1. Ranked actions + reason codes

The queue keeps the Week-4 baseline logic because its reasons are readable and available at the decision moment:

- +2 if content age is at least 180 days
- +2 if March impressions are at least 500
- +1 if content age is at least 365 days
- +1 if March impressions are at least 5,000

Action priority is then assigned as follows:

1. `REVIEW_NOW` — score 5–6; strong age/visibility reason
2. `REVIEW_NEXT` — score 4 and average position ≤20; stale, visible, and already near meaningful search visibility
3. `MONITOR` — score 4 but average position >20; the rule fires, but the immediate opportunity is less clear
4. `DEFER` — score below 4; does not satisfy both core conditions

This ranking does not claim that refreshes cause recovery. It identifies pages that look worth checking first.

In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy

import json
from pathlib import Path
from urllib.request import urlopen

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

AUDIT_URL = (
    "https://raw.githubusercontent.com/Cooper30/"
    "flyrank-ml-internship/main/work/outputs/w06_validation_audit.json"
)
with urlopen(AUDIT_URL) as response:
    validation_audit = json.load(response)

assert validation_audit["verdict"].startswith("LIMITED")
print("Model audit verdict:", validation_audit["verdict"])
print("Operational fallback: transparent rule + mandatory human review")

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Add your Hugging Face READ token to Colab Secrets as HF_TOKEN.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

pages = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_march,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march,
        COUNT(DISTINCT report_date) AS observed_days_march
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m.*,
    DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
FROM march m
LEFT JOIN {DIM_CONTENT} c
    ON m.content_hash_id = c.content_hash_id
WHERE c.content_created_date IS NOT NULL
  AND c.content_created_date <= DATE '2026-03-31'
""").df()

queue = pages.copy()
queue["baseline_action_score"] = 0
queue.loc[queue["content_age_days"] >= 180, "baseline_action_score"] += 2
queue.loc[queue["impressions_march"] >= 500, "baseline_action_score"] += 2
queue.loc[queue["content_age_days"] >= 365, "baseline_action_score"] += 1
queue.loc[queue["impressions_march"] >= 5000, "baseline_action_score"] += 1

conditions = [
    queue["baseline_action_score"] == 6,
    (queue["baseline_action_score"] == 5) & (queue["content_age_days"] >= 365),
    queue["baseline_action_score"] == 5,
    queue["baseline_action_score"] == 4,
]
reason_codes = [
    "VERY_STALE_HIGH_VOLUME",
    "VERY_STALE_VISIBLE",
    "STALE_HIGH_VOLUME",
    "STALE_VISIBLE",
]
queue["reason_code"] = np.select(conditions, reason_codes, default="BELOW_CORE_RULE")

queue["action"] = np.select(
    [
        queue["baseline_action_score"] >= 5,
        (queue["baseline_action_score"] == 4) & (queue["avg_position_march"] <= 20),
        queue["baseline_action_score"] == 4,
    ],
    ["REVIEW_NOW", "REVIEW_NEXT", "MONITOR"],
    default="DEFER",
)

action_order = {"REVIEW_NOW": 4, "REVIEW_NEXT": 3, "MONITOR": 2, "DEFER": 1}
queue["action_priority"] = queue["action"].map(action_order)
queue = queue.sort_values(
    ["action_priority", "baseline_action_score", "impressions_march", "content_age_days"],
    ascending=[False, False, False, False],
).reset_index(drop=True)
queue.insert(0, "action_rank", np.arange(1, len(queue) + 1))

display(queue[[
    "action_rank", "content_hash_id", "action", "reason_code",
    "baseline_action_score", "content_age_days", "impressions_march",
    "ctr_march", "avg_position_march"
]].head(20))


## 2. Intended use and limits

An editor uses this queue to decide where to begin manual content review. `REVIEW_NOW` means inspect first; it does not mean automatically rewrite, publish, redirect, merge, or delete. `REVIEW_NEXT` highlights stale and visible pages already within the top 20 average-position range. `MONITOR` keeps lower-visibility boundary cases visible without consuming immediate editorial capacity.

The queue is valid for the March 2026 observation window and the included portfolios. Content age is based on creation date rather than a verified last meaningful update, so an old but recently improved page may be ranked too highly. Search demand can also be seasonal, branded, or strategically important in ways these fields do not capture.

In [ ]:
action_summary = (
    queue.groupby("action", dropna=False)
    .size()
    .rename("pages")
    .reset_index()
)
action_summary["share"] = action_summary["pages"] / len(queue)
display(action_summary.sort_values("pages", ascending=False).round(4))

reason_summary = (
    queue.groupby("reason_code", dropna=False)
    .size()
    .rename("pages")
    .reset_index()
    .sort_values("pages", ascending=False)
)
display(reason_summary)


## 3. Human review + the no-go list

Before acting on a recommended page, the editor must check:

1. Whether the content is factually outdated or only old by creation date
2. Whether March demand is seasonal, branded, or event-driven
3. Whether the page still matches the intended search intent
4. Whether another page competes for the same topic
5. Whether the page has business, legal, or compliance importance
6. Whether a refresh, merge, redirect, protection, or monitoring action is actually appropriate

**Never automate:** publishing, deletion, pruning, redirecting, merging, or causal claims about refresh impact. The queue cannot verify factual correctness, brand strategy, or search intent on its own.

In [ ]:
review_checks = pd.DataFrame([
    ["Freshness", "Is the page actually outdated?", "Human"],
    ["Demand", "Is visibility seasonal, branded, or event-driven?", "Human"],
    ["Intent", "Does the page still answer the intended query?", "Human"],
    ["Overlap", "Is another page competing for the same topic?", "Human"],
    ["Action", "Refresh, protect, merge, redirect, or monitor?", "Human"],
], columns=["check", "question", "owner"])
display(review_checks)


## 4. Monitoring / retrain triggers

The queue should be regenerated monthly from the latest complete observation window. The model should not replace the rule until it beats both the rule baseline and the relevant base rate consistently across client-grouped evaluations.

Retrain or pause the scoring system when any of these occur:

- Client-level decline base rate moves by more than 10 percentage points
- Coverage or missingness for a key feature changes by more than 10 percentage points
- Manual acceptance of the top 50 recommendations falls below 50% for two cycles
- Precision@50 fails to beat both the rule and base rate in two consecutive evaluations
- The label, reporting window, or data collection process changes
- A new model depends mainly on measurement coverage rather than content/search signals

In [ ]:
monitoring = pd.DataFrame([
    ["Base-rate shift", ">10 percentage points", "Revalidate by client"],
    ["Feature coverage", ">10 percentage points", "Audit pipeline and missingness"],
    ["Top-50 manual acceptance", "<50% for two cycles", "Pause or revise rules"],
    ["Precision@50", "Below rule or base rate twice", "Do not promote model"],
    ["Label/data definition", "Any material change", "Rebuild and document"],
], columns=["signal", "trigger", "response"])
display(monitoring)


## 5. Exports for the paper

The full page-level queue is written to CSV for operational review and remains gitignored. A compact JSON receipt with aggregate action counts, reason-code counts, thresholds, limitations, and monitoring rules is safe to commit and use in the public paper.

In [ ]:
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_PATH = OUTPUT_DIR / "w07_action_queue.csv"
RECEIPT_PATH = OUTPUT_DIR / "w07_action_playbook.json"

queue.to_csv(QUEUE_PATH, index=False)

action_counts = {str(k): int(v) for k, v in queue["action"].value_counts().items()}
reason_counts = {str(k): int(v) for k, v in queue["reason_code"].value_counts().items()}

receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "observation_window": "2026-03",
    "total_pages_ranked": int(len(queue)),
    "operating_mode": "transparent rule plus mandatory human review",
    "model_audit_verdict": validation_audit["verdict"],
    "action_counts": action_counts,
    "reason_code_counts": reason_counts,
    "rule": {
        "minimum_age_days": 180,
        "minimum_impressions": 500,
        "very_stale_bonus_days": 365,
        "high_volume_bonus_impressions": 5000,
        "review_next_max_average_position": 20,
    },
    "automated_actions_allowed": False,
    "human_review_required": True,
    "limitations": [
        "content age is based on creation date, not verified last meaningful update",
        "March demand may be seasonal or portfolio-specific",
        "the rule does not establish causal refresh impact",
        "the Week-5 model did not beat the test base rate",
    ],
    "monitoring_triggers": monitoring.to_dict(orient="records"),
}

with open(RECEIPT_PATH, "w", encoding="utf-8") as f:
    json.dump(receipt, f, indent=2)

assert queue["action_rank"].is_unique
assert queue["action_rank"].min() == 1
assert queue["action_rank"].max() == len(queue)
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()
assert receipt["automated_actions_allowed"] is False
assert RECEIPT_PATH.exists()
assert QUEUE_PATH.exists()

print("ACTION PLAYBOOK PASSED ✅")
print(f"Pages ranked: {len(queue):,}")
print("Action counts:", action_counts)
print(f"Queue written to: {QUEUE_PATH} (gitignored)")
print(f"Receipt written to: {RECEIPT_PATH}")

from google.colab import files
files.download(str(RECEIPT_PATH))


## Self-check

After **Runtime → Run all** completes, confirm:

- [ ] One ranked page-level queue is generated
- [ ] Every row has one action and one reason code
- [ ] The model audit verdict is carried into the operating decision
- [ ] Intended use and limitations are explicit
- [ ] Human checks and prohibited automated actions are explicit
- [ ] Monitoring and pause/retrain triggers are defined
- [ ] No future-window or label-derived field is used for ranking
- [ ] `ACTION PLAYBOOK PASSED ✅` appears
- [ ] `w07_action_playbook.json` is downloaded for the committed receipt